In [28]:
import pandas as pd

def calculate_suicide_rates(suicides_csv, output_csv_name):
    """
    Calculates the suicide rate per lakh population for each state in India for the year 2011,
    based on the provided suicide data and 2011 census population data.

    Args:
        suicides_csv (str): The filename for the suicide data.
        output_csv_name (str): The filename for the final output CSV.
    """
    # 1. Load the suicide data
    try:
        suicide_df = pd.read_csv(suicides_csv)
    except FileNotFoundError:
        print(f"Error: The file '{suicides_csv}' was not found.")
        return

    # 2. Correctly calculate total suicides for 2011 to avoid double-counting
    # Filter for the year 2011 and a non-overlapping category like 'Social_Status'
    suicides_2011 = suicide_df[
        (suicide_df['Year'] == 2011) & (suicide_df['Type_code'] == 'Social_Status')
    ]
    suicide_statewise = suicides_2011.groupby('State', as_index=False)['Total'].sum()
    suicide_statewise.rename(columns={'Total': 'Total_Suicides'}, inplace=True)
    suicide_statewise['State'] = suicide_statewise['State'].str.strip()

    # 3. Get 2011 population data for the calculation
    # This data was manually obtained from the 2011 Census of India
    population_data = {
        'State': [
            'Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar',
            'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh',
            'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Lakshadweep',
            'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram',
            'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan',
            'Sikkim', 'Tamil Nadu', 'Tripura', 'Uttar Pradesh', 'Uttarakhand',
            'West Bengal', 'Delhi', 'Dadra and Nagar Haveli and Daman and Diu'
        ],
        'Population': [
            380581, 84580777, 1383727, 31205576, 104099452,
            25545198, 1458545, 60439692, 25351462, 6864602,
            12541302, 32988134, 61095297, 33406061, 64473,
            72626809, 112374333, 2855794, 2966889, 1097206,
            1978502, 41974218, 1247953, 27743338, 68548437,
            610577, 72147030, 3673917, 199812341, 10086292,
            91276115, 16787941, 586956
        ]
    }
    population_df = pd.DataFrame(population_data)

    # 4. Standardize state names for merging
    state_mapping = {
        "A & N Islands": "Andaman and Nicobar Islands",
        "Andaman and Nicobar Islands[UT]": "Andaman and Nicobar Islands",
        "NCT of Delhi": "Delhi",
        "NCT (Delhi)": "Delhi",
        "Delhi[UT]": "Delhi",
        "Jammu & Kashmir": "Jammu and Kashmir",
        "Jammu & Kashmir[UT]": "Jammu and Kashmir",
        "Puducherry[UT]": "Puducherry",
        "Dadra & Nagar Haveli": "Dadra and Nagar Haveli and Daman and Diu",
        "Daman & Diu": "Dadra and Nagar Haveli and Daman and Diu",
        "Uttaranchal": "Uttarakhand",
        "Orissa": "Odisha",
    }

    def clean_state_column(df, col):
        df[col] = df[col].replace(state_mapping)
        df[col] = df[col].str.strip()
        return df

    suicide_statewise = clean_state_column(suicide_statewise, "State")
    population_df = clean_state_column(population_df, "State")

    # 5. Merge the dataframes
    merged_df = pd.merge(suicide_statewise, population_df, on='State', how='left')

    # Fill NaN values in the Total_Suicides column with 0
    merged_df['Total_Suicides'] = merged_df['Total_Suicides'].fillna(0)

    # 6. Calculate the corrected suicide rate per lakh
    merged_df['Suicide_Rate_Per_Lakh'] = (
        merged_df['Total_Suicides'] / merged_df['Population'] * 100000
    )

    # Save the final merged dataset to CSV
    merged_df.to_csv(output_csv_name, index=False)
    print(f"Final dataset saved to {output_csv_name}")

# Example usage
if __name__ == "__main__":
    calculate_suicide_rates(
        suicides_csv='suicides.csv',
        output_csv_name='final_corrected_dataset.csv'
    )


Final dataset saved to final_corrected_dataset.csv
